# 04 — Testing Against Multiple Features

One point, one polygon — that's a toy problem. A real query is:

```text
Given a point, which of these N polygons contains it?
```

This is just a loop. But the loop has two important variants with different use cases:

- **Early exit** — stop at the first match (fast, assumes non-overlapping regions)
- **Full scan** — test every polygon, collect all matches (required for overlapping regions)

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)


def point_in_ring(lon, lat, ring):
    """Ray casting: True if (lon, lat) is inside the closed ring."""
    n = len(ring) - 1
    inside = False
    for i in range(n):
        xi, yi = ring[i][0],   ring[i][1]
        xj, yj = ring[i+1][0], ring[i+1][1]
        if (yi > lat) != (yj > lat):
            x_intersect = xi + (lat - yi) * (xj - xi) / (yj - yi)
            if lon < x_intersect:
                inside = not inside
    return inside


print(f"Loaded {len(regions_fc['features'])} regions.")

## Testing One Point Against One Feature

A GeoJSON Polygon geometry has `coordinates` as a list of rings — the first is the exterior ring, subsequent ones (if any) are holes. For now, we only deal with the exterior ring.

In [ ]:
def point_in_feature(lon, lat, feature):
    """
    Test (lon, lat) against a single GeoJSON Feature.
    Handles Polygon geometry only (exterior ring).
    Returns True if the point is inside.
    """
    geom = feature["geometry"]
    if geom["type"] != "Polygon":
        return False
    exterior_ring = geom["coordinates"][0]
    return point_in_ring(lon, lat, exterior_ring)


# Test Tehran against every sector
tehran_lon, tehran_lat = 51.388, 35.695

print("Tehran against each sector:")
for feat in regions_fc["features"]:
    name   = feat["properties"]["name"]
    result = point_in_feature(tehran_lon, tehran_lat, feat)
    print(f"  {name:<20}  inside={result}")

## Early Exit: Find the First Match

If regions are non-overlapping (like our sectors), a point can be in at most one region. Return as soon as you find it.

In [ ]:
def find_containing_feature(lon, lat, feature_collection):
    """
    Return the FIRST feature in the FeatureCollection that contains (lon, lat).
    Returns None if no feature contains the point.
    Early exit: stops searching as soon as a match is found.
    """
    for feature in feature_collection["features"]:
        if point_in_feature(lon, lat, feature):
            return feature
    return None


# Test all target cities
cities = [
    ("Tehran",   51.388,  35.695),
    ("Riyadh",   46.675,  24.688),
    ("Cairo",    31.235,  30.044),
    ("Madrid",   -3.703,  40.417),
    ("Muscat",   58.593,  23.614),
    ("Moscow",   37.617,  55.755),
]

print(f"{'City':<12}  {'Sector found':>25}")
print("─" * 42)
for name, lon, lat in cities:
    match = find_containing_feature(lon, lat, regions_fc)
    sector = match["properties"]["name"] if match else "(none — outside all sectors)"
    print(f"  {name:<10}  {sector:>26}")

## Full Scan: Find All Matches

When regions can overlap — blast zones, coverage areas, political zones with contested boundaries — a point might be in multiple regions simultaneously. Full scan returns all of them.

In [ ]:
def find_all_containing_features(lon, lat, feature_collection):
    """
    Return ALL features that contain (lon, lat).
    Returns empty list if no feature contains the point.
    """
    return [
        feat for feat in feature_collection["features"]
        if point_in_feature(lon, lat, feat)
    ]


# Build an overlapping test dataset — two overlapping blast zones
import math

def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def make_circle(lon, lat, radius_km, n=64):
    ring = [destination_point(lon, lat, 360.0*i/n, radius_km) for i in range(n)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

# Tehran blast zone (500 km) and Riyadh blast zone (500 km) — they overlap in the Gulf
tehran_zone = {
    "type": "Feature",
    "properties": {"name": "Tehran 500 km zone"},
    "geometry": make_circle(51.388, 35.695, 500)
}
riyadh_zone = {
    "type": "Feature",
    "properties": {"name": "Riyadh 500 km zone"},
    "geometry": make_circle(46.675, 24.688, 500)
}
overlapping_fc = {"type": "FeatureCollection", "features": [tehran_zone, riyadh_zone]}

# Test a point in the overlap zone (roughly between Iran and Saudi Arabia)
overlap_lon, overlap_lat = 50.0, 28.0   # Kuwait area

matches = find_all_containing_features(overlap_lon, overlap_lat, overlapping_fc)
print(f"Point ({overlap_lon}, {overlap_lat}) is inside {len(matches)} zone(s):")
for m in matches:
    print(f"  → {m['properties']['name']}")

## Performance: Bounding Box Pre-Filter

For datasets with many polygons, the ray casting test itself is not the bottleneck — but calling it on polygons that the point can't possibly be in wastes time. A bbox pre-check eliminates obvious misses in O(1) before doing the full edge loop.

In [ ]:
def bbox_of_ring(ring):
    """Return (min_lon, min_lat, max_lon, max_lat) for a ring."""
    lons = [p[0] for p in ring]
    lats = [p[1] for p in ring]
    return min(lons), min(lats), max(lons), max(lats)


def point_in_feature_fast(lon, lat, feature):
    """Point-in-polygon with bounding box pre-check."""
    geom = feature["geometry"]
    if geom["type"] != "Polygon":
        return False
    ring = geom["coordinates"][0]
    min_lon, min_lat, max_lon, max_lat = bbox_of_ring(ring)
    # Fast reject
    if lon < min_lon or lon > max_lon or lat < min_lat or lat > max_lat:
        return False
    return point_in_ring(lon, lat, ring)


# Count bbox skips for points far outside the sectors
far_away_cities = [
    ("Honolulu", -157.855, 21.305),
    ("New York",  -74.006, 40.712),
    ("Beijing",   116.391, 39.904),
]

print("Bbox pre-check on far-away cities (should skip all 5 sectors):")
for city, lon, lat in far_away_cities:
    skipped = 0
    for feat in regions_fc["features"]:
        ring = feat["geometry"]["coordinates"][0]
        mn_lon, mn_lat, mx_lon, mx_lat = bbox_of_ring(ring)
        if lon < mn_lon or lon > mx_lon or lat < mn_lat or lat > mx_lat:
            skipped += 1
    print(f"  {city:<12}  skipped {skipped}/5 sectors via bbox")

## Exercise A

Write a function `which_sector(lon, lat)` that takes a point and returns the name of the sector it falls in (or `"none"`).

1. Use `find_containing_feature` (early exit) internally.
2. Test it with Tehran, Riyadh, Cairo, Madrid, and Muscat.
3. Print a clean table: city name, lon, lat, sector result.

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")
with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)

def which_sector(lon, lat):
    # Use find_containing_feature, return sector name or "none"
    pass  # your code here

cities = [
    ("Tehran",  51.388,  35.695),
    ("Riyadh",  46.675,  24.688),
    ("Cairo",   31.235,  30.044),
    ("Madrid",  -3.703,  40.417),
    ("Muscat",  58.593,  23.614),
]

# Print: city name, lon, lat, which_sector result
# Your code here

## Exercise B

Create **three overlapping blast zones** (800 km each) around Tehran, Riyadh, and Cairo.

1. Build them as a FeatureCollection using `make_circle`.
2. Use `find_all_containing_features` to test which zones contain each of these points:
   - `(40.0, 30.0)` — Red Sea area
   - `(50.0, 25.0)` — Gulf of Oman
   - `(20.0, 28.0)` — Sudan
3. For each test point, print how many zones it falls in and which ones.

In [ ]:
import math

# make_circle(lon, lat, radius_km) — defined in teaching cells above

# 1. Three 800 km blast zones: Tehran, Riyadh, Cairo
# 2. find_all_containing_features for each test point
# 3. Print zone membership for each point
# Your code here

---

## Check Your Understanding

**1.** When would you use early exit vs full scan, and what does each assume about the data?

**2.** Why is bounding box pre-filtering most useful when the dataset has many polygons and the test point is far from most of them?

```python
# No code needed — answer in your own words
```

## Next

In [05 — Region Classification](./05_Region_Classification.ipynb), we extract properties from the matched feature to turn a geometric result into a meaningful answer: not just "inside" but "inside **Sector Alpha** (Northern Gulf / Iran)".